# First audit — smallest thing that can succeed

**Goal: prove the pipeline runs end to end.** Not to find a vulnerability, not to
measure anything. Success is cell 6 printing a report — *even one with zero findings*.

Target is `eval/fixtures/toy_vuln`: 4 files, ~640 tokens, with two deliberately
planted cross-file vulnerabilities whose exact locations are recorded in
`GROUND_TRUTH.json`. So you can tell instantly whether output is real or invented.

At 640 tokens the memory problem that killed earlier runs cannot occur. Every
variable is removed except the one open question: **does the model emit parseable JSON?**

**Before starting: Runtime → Change runtime type → T4 GPU, then Runtime → Restart session.**

In [ ]:
#@title 1. Install — MUST print 0.3.0
import sys
assert "wca" not in sys.modules, "Stale module. Runtime > Restart session, then re-run."

REPO = "https://github.com/quinyang/whole_codebase_auditor"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

!pip install -q "wca[gpu] @ git+{REPO}@{BRANCH}"

import wca
print("wca version:", wca.__version__)
assert wca.__version__ >= "0.3.0", (
    f"Got {wca.__version__}, need >=0.3.0. The push did not land. On your laptop:\n"
    "  git add -A && git commit -m wip && git push origin rework && git push origin rework:main\n"
    "  git ls-remote --heads origin   # main and rework must show the same sha"
)
print("OK — new code is loaded.")

In [ ]:
#@title 2. Hardware check
from wca.infer import describe_environment
print(describe_environment())

# Expected on a free T4:
#   device: Tesla T4          <- if this says CPU, set Runtime > Change runtime type > T4
#   compute dtype: torch.float16   <- fp16, not bf16 (T4 is Turing, no real bf16)
#   mamba_ssm: absent (eager path)  <- fine at this size

In [ ]:
#@title 3. CPU stages — ingest → parse → graph → pack (no GPU, seconds)
import json, os

!rm -rf /content/wca_repo && git clone -q {REPO} -b {BRANCH} /content/wca_repo
FIXTURE = "/content/wca_repo/eval/fixtures/toy_vuln"
GT = json.load(open(f"{FIXTURE}/GROUND_TRUTH.json"))

from wca.ingest import ingest
from wca.parse import parse_files, LanguageDispatcher
from wca.graph import build_graph
from wca.pack import pack

bundle = ingest(FIXTURE)
parsed = parse_files(bundle.files, LanguageDispatcher())
graph  = build_graph(parsed.files)
packed = pack(parsed.files, graph, budget_tokens=4000, repo_name="toy_vuln")

print(bundle.summary()); print(parsed.summary())
print(graph.summary());  print(packed.stats_line())

# The answer key must not reach the model, or the whole exercise is worthless.
assert "GROUND_TRUTH" not in packed.text, "answer key leaked into the context!"
assert "admin_password_123" in packed.text, "planted secret missing from context"
print("\nOK — answer key excluded, planted secret present.")

print("\n--- what the model will see (first 700 chars) ---")
print(packed.text[:700])

In [ ]:
#@title 4. Load the model — watch the 'allocated' number
from wca.infer import MambaAuditor, free_gpu, estimate_max_context, fast_path_available

if "auditor" in globals():
    auditor.free(); del auditor
free_gpu()

auditor = MambaAuditor()   # first run downloads ~7 GB; later runs use the cache

# allocated ~6 GiB  -> 4-bit worked, good
# allocated >8 GiB  -> stale model resident; Runtime > Restart session
print("\nfused kernels:", fast_path_available())
print("safe context on this GPU:", f"{estimate_max_context(auditor.model):,} tokens")
print("this audit needs:", f"{packed.used_tokens:,} tokens")

In [ ]:
#@title 5. Generate — the moment of truth
# Repack with the real tokenizer so the count is exact rather than estimated.
packed = pack(parsed.files, graph, budget_tokens=4000,
              tokenizer=auditor.tokenizer, repo_name="toy_vuln")
print(packed.stats_line())

gen = auditor.generate(packed.text, max_new_tokens=512)
print(gen.stats_line())

print("\n=============== RAW MODEL OUTPUT ===============")
print(gen.text)
print("================================================")
# Read this yourself before trusting any parsing below. The open question for
# session 2 is whether this is valid JSON -- if it is prose, that is a real
# result, not a failure. Record it either way.

In [ ]:
#@title 6. Parse, ground, and score against the planted answers
from wca.findings import parse_findings, AuditReport

findings = parse_findings(gen.text, packed)
report = AuditReport(
    repo="toy_vuln", model=auditor.model_id, findings=findings,
    pack_stats={"budget_tokens": packed.budget_tokens, "used_tokens": packed.used_tokens},
    gen_stats={"prompt_tokens": gen.prompt_tokens, "seconds": round(gen.total_seconds, 1)},
)
print(report.pretty())

# --- did it find what was actually planted? -------------------------------
print("\n=== vs GROUND_TRUTH ===")
hit_files = {f for fi in report.grounded for f in fi.files}
for p in GT["planted"]:
    need = set(p["requires_files"])
    status = "FOUND (both files)" if need <= hit_files else (
             "partial" if need & hit_files else "missed")
    print(f"  {p['id']:15} [{p['severity']:8}] {status}")

print(f"\ntotal={len(findings)}  grounded={len(report.grounded)}  "
      f"cross-file={len(report.cross_file)}")
if findings and not report.grounded:
    print("\nAll findings ungrounded -> the model is inventing evidence lines. "
          "That is a genuine result worth recording.")

os.makedirs("/content/wca_runs", exist_ok=True)
report.save("/content/wca_runs/toy_vuln.findings.json")
open("/content/wca_runs/toy_vuln.raw.txt", "w").write(gen.text)
print("\nsaved -> /content/wca_runs/  (download these; they are session 2's input)")

## How to read the result

| Outcome | Meaning | Next |
|---|---|---|
| Report prints, findings grounded, both planted issues found | Best case | Session 2: scale context up |
| Report prints, findings grounded, planted issues missed | Pipeline works, prompt or model is weak | Session 2: iterate on the prompt |
| Report prints, all findings **ungrounded** | Model invents evidence lines | Session 2: tighten the schema; record the hallucination rate |
| Zero findings parsed, raw output is prose | Model ignored the JSON schema | Session 2: shorten schema, move instructions after the code |
| Zero findings, raw output is empty | Generation problem | Check `gen.stats_line()`, raise `max_new_tokens` |

**All five are successes for this session.** You are establishing a baseline, and
the pipeline ran either way. Only a traceback is a failure.

Record the raw output — it is the input to session 2's decision about whether the
JSON schema survives at longer context.